In [22]:
import ast
import csv
from distutils.log import error 
import json
import math
import numpy as np

import led_config_utils
import mesh_config
from funky_lights import led_config

from importlib import reload
reload(mesh_config)

START_OFFSET = 0.0
END_OFFSET = 0.0

# Collapse some segments
SEGMENT_1 = [1,2,3,4]
SEGMENT_2 = [5,6,7,8]
SEGMENT_3 = [9,10,11,12,13,14]
SEGMENT_4 = [15,16,17,18,19,20]
SEGMENT_5 = [21,22,23,24,25,26,27,28,29,30,31,32]

def Rx(theta):
  return np.matrix([[ 1,                 0,                0],
                    [ 0,   math.cos(theta), -math.sin(theta)],
                    [ 0,   math.sin(theta),  math.cos(theta)]])

def Ry(theta):
  return np.matrix([[ math.cos(theta), 0, math.sin(theta)],
                   [                0, 1,               0],
                   [ -math.sin(theta), 0, math.cos(theta)]])

def Rz(theta):
  return np.matrix([[ math.cos(theta), -math.sin(theta), 0],
                    [ math.sin(theta),  math.cos(theta), 0],
                    [               0,                0, 1]])

def createNodesFromCSV(csv_points, R, uid):
    nodes = None
    prev_node = None
    for point in csv_points:
        print(point)
        point = np.array(point)
        point = (R * point.reshape((3,1))).reshape((1,3))
        point = np.squeeze(np.asarray(point))
        # Hack to flip the headboard segments vertically. The connection in on the bottom right, not the top left.
        if uid in SEGMENT_5:
            point[1] = -(point[1] - 1.4) + 1.4
        node = led_config_utils.Node(p=point)
        if nodes == None:
            nodes = node
        if prev_node:
            prev_node.next = node
        prev_node = node
    return nodes



all_segments = {}
with open('../config/bed_actual.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    segments=[]

    for row in reader:
        if len(row) == 0 or row[0].startswith('#'):
            continue
        uid = int(row[0])
        name = row[1]
        actual_num_leds = int(row[2])
        actual_length = float(row[3])
        reverse = (row[4] == 'TRUE')
        led_offset = int(row[5])
        sub_component = row[6]
        actual_num_adressable_leds = int(row[7])
        csv_points = ast.literal_eval(row[10])

        # Rotate points around Y to match the model orientation
        R = Ry(math.radians(90))
        nodes = createNodesFromCSV(csv_points, R, uid)
        if nodes == None:
            print('No nodes available.')
            continue
    
        modelled_length = led_config_utils.line_segments_length(nodes)
        modelled_length = modelled_length -  START_OFFSET -  END_OFFSET
        
        leds_distance = modelled_length / (actual_num_adressable_leds - 0.999) # -0.999 to avoid rounding issues with the last LED
        points = led_config_utils.trace_line_segments(nodes, actual_num_adressable_leds, START_OFFSET, leds_distance)
        if len(points) == 0:
            print('No points available.')
            continue

        if reverse:
            points = np.flip(points, axis=0)

        if led_offset > 0:
            points = np.concatenate((points[led_offset:], points[:led_offset]), axis=0)

        segment = led_config.Segment(
            uid=uid, name=name, points=points, num_leds=points.shape[0], length=actual_length)
        print('Segment %s: length=%.1fm, num_leds=%s' % (segment.name, segment.length, segment.num_leds))
        segments.append(segment)
        all_segments[uid] = segment



for merge_list in [SEGMENT_1, SEGMENT_2, SEGMENT_3, SEGMENT_4, SEGMENT_5]:
    merged_segment = all_segments[merge_list[0]]
    for uid in merge_list[1:]:
        segment = all_segments.pop(uid)
        merged_segment.merge(segment)

# Create LED config
config = led_config.LedConfig()
for segment in all_segments.values():
    config.led_segments.append(segment)
    config.total_num_segments += 1
    config.total_length += segment.length
    config.total_num_leds += segment.num_leds

with open('../config/led_config_bed.json', 'w', encoding='utf-8') as f:
    json.dump(config.to_dict(), f, ensure_ascii=False, indent=4)

[0.766, 0.609, 0.981]
[0.766, 0.0, 0.981]
Segment frame: length=0.6m, num_leds=12
[0.726, 0.0, 1.01]
[0.726, 0.609, 1.01]
Segment frame: length=0.6m, num_leds=12
[0.726, 0.94, 1.01]
[0.726, 1.981, 1.01]
Segment frame: length=1.0m, num_leds=21
[0.766, 1.981, 0.981]
[0.766, 0.94, 0.981]
Segment frame: length=1.0m, num_leds=21
[-0.766, 0.609, 0.981]
[-0.766, 0.0, 0.981]
Segment frame: length=0.6m, num_leds=12
[-0.726, 0.0, 1.01]
[-0.726, 0.609, 1.01]
Segment frame: length=0.6m, num_leds=12
[-0.726, 0.94, 1.01]
[-0.726, 1.981, 1.01]
Segment frame: length=1.0m, num_leds=21
[-0.766, 1.981, 0.981]
[-0.766, 0.94, 0.981]
Segment frame: length=1.0m, num_leds=21
[0.766, 0.609, -0.981]
[0.766, 0.0, -0.981]
Segment frame: length=0.6m, num_leds=12
[0.726, 0.0, -1.01]
[0.726, 0.609, -1.01]
Segment frame: length=0.6m, num_leds=12
[0.752, 0.94, -1.077]
[0.752, 1.981, -1.077]
Segment frame: length=1.0m, num_leds=21
[0.721, 1.981, -1.037]
[0.721, 0.94, -1.037]
Segment frame: length=1.0m, num_leds=21
[0.7